# Supervised Learning Comparison

This Jupyter Notebook analyzes breast cancer and Titanic data using five classification algorithms: SVM, Decision Tree, Boosting Tree, Neural Network, and KNN. For each model, there will be a learning curve and model complexity plots after tuning is performed.

Warning - Depending on your computer, this can take a long time to load. You make also need to install (ex. pip install pandas) the necessary libraries shown below.

# Import Libraries

In [ ]:
import pandas as da
import random
import warnings
import numpy as my
%matplotlib inline
import matplotlib.pyplot as ppt
ppt.style.use("seaborn-whitegrid")
warnings.filterwarnings("ignore")
from sklearn.model_selection import GridSearchCV, train_test_split, cross_validate
import timeit
from sklearn.metrics import accuracy_score, roc_auc_score, precision_score, f1_score, recall_score
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
my.random.seed(7)
from sklearn.neighbors import KNeighborsClassifier as KNC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn import preprocessing
import itertools


## Load and Process Breast Cancer Data

In [ ]:
bc_df = da.read_csv("breastcancer.csv")
co = list(bc_df)
co.insert(0, co.pop(co.index("diagnosis")))
bc_df["diagnosis"].replace("M",1,inplace=True)
bc_df = bc_df.loc[:, co]
bc_df["diagnosis"].replace("B",0,inplace=True)
bc_df["diagnosis"] = bc_df["diagnosis"].astype("category")


## Load and Process Titanic Data

In [ ]:
tt_df = da.read_csv("titanic.csv")
tt_df["Survived"] = tt_df["Survived"].astype("category")

In [ ]:
bcx = my.array(bc_df.values[:,1:],dtype="int64")
bcx = preprocessing.scale(bcx)
bcy = my.array(bc_df.values[:,0],dtype="int64")
ttx = my.array(tt_df.values[:,1:],dtype="int64")
tty = my.array(tt_df.values[:,0],dtype="int64")

bc_xtn, bc_xtt, bc_ytn, bc_ytt = train_test_split(my.array(bcx),my.array(bcy), test_size=0.20, random_state=7)
tt_xtn, tt_xtt, tt_ytn, tt_ytt = train_test_split(my.array(ttx),my.array(tty), test_size=0.20, random_state=7)


## Cross-Classifier Functions

In [ ]:
ppt.rcParams["xtick.labelsize"] = 11
ppt.rcParams["ytick.labelsize"] = 11
ppt.rcParams["axes.titlesize"] = 11
ppt.rcParams["font.size"] = 11

def pltlc(clss, inp, out, ttl="Title"):
    cvavg = []; cvstd = []; tnavg = []; tnstd = []
    prdavg = []; prdstd = []
    ftavg = []; ftstd = [] 
    tnsz= (my.linspace(.05, 1.0, 20)*(len(out))).astype("int")  
    
    for s in tnsz:
        ix = my.random.randint(inp.shape[0], size=s)
        scrs = cross_validate(clss, inp[ix,:], out[ix], cv=10, scoring="f1", return_train_score=True, n_jobs=-1)
        
        ftavg.append(my.mean(scrs["fit_time"])); ftstd.append(my.std(scrs["fit_time"]))
        tnavg.append(my.mean(scrs["train_score"])); tnstd.append(my.std(scrs["train_score"]))
        prdavg.append(my.mean(scrs["score_time"])); prdstd.append(my.std(scrs["score_time"]))
        cvavg.append(my.mean(scrs["test_score"])); cvstd.append(my.std(scrs["test_score"]))
    
    tnavg = my.array(tnavg); tnstd = my.array(tnstd)
    cvavg = my.array(cvavg); cvstd = my.array(cvstd)
    prdavg = my.array(prdavg); prdstd = my.array(prdstd)
    ftavg = my.array(ftavg); ftstd = my.array(ftstd)
    
    lc_plt(tnsz, tnavg, tnstd, cvavg, cvstd, ttl)
    plt_tms(tnsz, ftavg, ftstd, prdavg, prdstd, ttl)
    
    return tnsz, tnavg, ftavg, prdavg
    

def lc_plt(tnsz, tnavg, tnstd, cvavg, cvstd, ttl):
    
    ppt.figure()
    ppt.xlabel("Training Ex [#]")
    ppt.title("Learning Curve: "+ ttl)
    ppt.fill_between(tnsz, tnavg - 2*tnstd, tnavg + 2*tnstd, alpha=0.1, color="y")
    ppt.plot(tnsz, tnavg, "o-", color="y", label="Training")
    ppt.fill_between(tnsz, cvavg - 2*cvstd, cvavg + 2*cvstd, alpha=0.1, color="m")
    ppt.plot(tnsz, cvavg, "o-", color="m", label="Cross-Validation")
    ppt.legend(loc="best")
    ppt.ylabel("F1 Score")
    ppt.show()
    
    
def plt_tms(tnsz, ftavg, ftstd, prdavg, prdstd, ttl):
    
    ppt.figure()
    ppt.xlabel("Training Ex [#]")
    ppt.title("Modeling: "+ ttl)
    ppt.fill_between(tnsz, ftavg - 2*ftstd, ftavg + 2*ftstd, alpha=0.1, color="y")
    ppt.plot(tnsz, ftavg, "o-", color="y", label="Training Time")
    ppt.fill_between(tnsz, prdavg - 2*prdstd, prdavg + 2*prdstd, alpha=0.1, color="m")
    ppt.plot(tnsz, prdavg, "o-", color="m", label="Prediction Time")
    ppt.legend(loc="best")
    ppt.ylabel("Time [s]")
    ppt.show()
    
def class_eval(clss, xtrn, xtst, ytrn, ytst):
    
    start = timeit.default_timer()
    clss.fit(xtrn, ytrn)
    end = timeit.default_timer()
    trn_tm = end - start
    
    start = timeit.default_timer()    
    yprd = clss.predict(xtst)
    end = timeit.default_timer()
    prd_tm = end - start

    print("Evaluation Metrics on Test Data")
    print("--------------------------------")
    print("Recall: "+"{:.2f}".format(recall_score(ytst, yprd)))
    print("Precision: "+"{:.2f}".format(precision_score(ytst, yprd)))
    print("Accuracy:  "+"{:.2f}".format(accuracy_score(ytst, yprd)))
    print("     AUC:  "+"{:.2f}".format(roc_auc_score(ytst, yprd)))
    print("F1 Score:  "+"{:.2f}".format(f1_score(ytst, yprd)))
    print("Training Time [s]:   "+"{:.5f}".format(trn_tm))
    print("Prediction Time [s]: "+"{:.5f}\n".format(prd_tm))

# Support Vector Machine Classifier

The main SVM hyperparameter will be the kernel function: linear, radial basis function (rbf), or sigmoid. The penalty term C and the kernel coefficient gamma will also be explored.

In [ ]:
def hsvm(xtn, ytn, xtst, ytst, ttl):
    xvl = ["linear","rbf","sigmoid"]
    f1tn = []; f1tst = []
    for v in xvl:            
            clss = SVC(kernel=v, random_state=7)
            clss.fit(xtn, ytn)
            f1tst.append(f1_score(ytst, clss.predict(xtst)))
            f1tn.append(f1_score(ytn, clss.predict(xtn)))
                
    ppt.plot(xvl, f1tst, "o-", color="m", label="Test")
    ppt.plot(xvl, f1tn, "o-", color = "y", label="Train")
    ppt.ylabel("F1 Score")
    ppt.xlabel("Kernel Function")
    
    ppt.title(ttl)
    ppt.legend(loc="best")
    ppt.tight_layout()
    ppt.show()
    
def SVMGSCV(xtn, ytn, kernel):
    param_grid = {"C": [0.1, 1, 10], "gamma": [0.1, 1, 10, 100]}
    clss = GridSearchCV(cv=10, estimator = SVC(kernel=kernel, random_state=7),
                       param_grid=param_grid)
    clss.fit(xtn, ytn)
    print("Best Parameters -")
    print(clss.best_params_)
    return clss.best_params_["C"], clss.best_params_["gamma"]

In [ ]:
#BC
hsvm(bc_xtn, bc_ytn, bc_xtt, bc_ytt, ttl="SVM Complexity Curve (Breast Cancer)\nHyperparameter : Kernel Function")

cval, gammavl = SVMGSCV(bc_xtn, bc_ytn, kernel="linear")

estmtr_bc = SVC(C=cval, gamma=gammavl, kernel="linear", random_state=7)

trn_sampsiz_bc, SVM_tn_scr_bc, SVM_ft_tm_bc, SVM_prd_time_bc = pltlc(estmtr_bc, bc_xtn, bc_ytn, ttl="SVM Breast Cancer")
class_eval(estmtr_bc, bc_xtn, bc_xtt, bc_ytn, bc_ytt)


# Titanic
hsvm(tt_xtn, tt_ytn, tt_xtt, tt_ytt, ttl="SVM Complexity Curve (Titanic)\nHyperparameter : Kernel Function")

cval, gammavl = SVMGSCV(tt_xtn, tt_ytn, kernel="rbf")

estmtr_tt = SVC(C=cval, gamma=gammavl, kernel="rbf", random_state=7)

estmtr_tt = SVC(kernel="rbf", random_state=7)

trn_sampsiz_tt, SVM_tn_scr_tt, SVM_ft_tm_tt, SVM_prd_time_tt = pltlc(estmtr_tt, tt_xtn, tt_ytn, ttl="SVM Titanic")
class_eval(estmtr_tt, tt_xtn, tt_xtt, tt_ytn, tt_ytt)


# Neural Network

This section builds a forward-feed network that computes weights using backpropagation. The parameter variables that will be tested are the number of hidden nodes and learning rate.

In [ ]:
def hnn(xtn, ytn, xtst, ytst, ttl):
    f1tn = []; f1tst = []
    hlist = my.linspace(1,100,30).astype("int")
    for h in hlist:         
            clss = MLPClassifier(solver="adam", hidden_layer_sizes=(h,), activation="logistic", 
                                learning_rate_init=0.05, random_state=7)
            clss.fit(xtn, ytn)
            f1tst.append(f1_score(ytst, clss.predict(xtst)))
            f1tn.append(f1_score(ytn, clss.predict(xtn)))
      
    ppt.plot(hlist, f1tst, "o-", color="m", label="Test")
    ppt.plot(hlist, f1tn, "o-", color = "y", label="Train")
    ppt.ylabel("F1 Score")
    ppt.xlabel("No. Hidden Units")
    
    ppt.title(ttl)
    ppt.legend(loc="best")
    ppt.tight_layout()
    ppt.show()
    
    
def NNGSCV(xtn, ytn):
    param_grid = {"hidden_layer_sizes": [5, 25, 50, 100], "learning_rate_init": [0.01, 0.05, 0.1]}
    nn = GridSearchCV(cv=10, estimator = MLPClassifier(solver="adam",activation="logistic",random_state=7),
                       param_grid=param_grid)
    nn.fit(xtn, ytn)
    print("Best Parameters -")
    print(nn.best_params_)
    return nn.best_params_["hidden_layer_sizes"], nn.best_params_["learning_rate_init"]


In [ ]:
# BC
hnn(bc_xtn, bc_ytn, bc_xtt, bc_ytt, ttl="NN Complexity Curve (Breast Cancer)\nHyperparameter : No. Hidden Units")

hunits, lrn_rt = NNGSCV(bc_xtn, bc_ytn)

estmtr_bc = MLPClassifier(hidden_layer_sizes=(hunits,), solver="adam", activation="logistic", 
                               learning_rate_init=lrn_rt, random_state=7)

trn_sampsiz_bc, NN_tn_scr_bc, NN_ft_tm_bc, NN_prd_time_bc = pltlc(estmtr_bc, bc_xtn, bc_ytn, ttl="NN Breast Cancer")
class_eval(estmtr_bc, bc_xtn, bc_xtt, bc_ytn, bc_ytt)



# Titanic
hnn(tt_xtn, tt_ytn, tt_xtt, tt_ytt, ttl="NN Complexity Curve (Titanic)\nHyperparameter : No. Hidden Units")

hunits, lrn_rt = NNGSCV(tt_xtn, tt_ytn)

estmtr_tt = MLPClassifier(hidden_layer_sizes=(hunits,), solver="adam", activation="logistic", 
                               learning_rate_init=lrn_rt, random_state=7)

trn_sampsiz_tt, NN_tn_scr_tt, NN_ft_tm_tt, NN_prd_time_tt = pltlc(estmtr_tt, tt_xtn, tt_ytn, ttl="NN Titanic")
class_eval(estmtr_tt, tt_xtn, tt_xtt, tt_ytn, tt_ytt)


In [ ]:
# BC
estmtr_bc = MLPClassifier(solver="adam", hidden_layer_sizes=(5,), activation="logistic", 
                               learning_rate_init=0.1, random_state=7)
estmtr_bc.fit(bc_xtn, bc_ytn)
b = estmtr_bc.loss_curve_

# Titanic
estmtr_tt = MLPClassifier(solver="adam", hidden_layer_sizes=(50,), activation="logistic", 
                               learning_rate_init=0.1, random_state=7)
estmtr_tt.fit(tt_xtn, tt_ytn)
t = estmtr_tt.loss_curve_

ppt.figure()
ppt.xlabel("Iterations")
ppt.title("Loss Curve")
ppt.plot(t, "o-", color="m", label="Titanic")
ppt.plot(b, "o-", color="y", label="Breast Cancer")
ppt.legend(loc="best")
ppt.ylabel("Loss")
ppt.show()

# Decision Tree

This section builds a decision tree using entropy-based information gain to discover the optimal feature split for ID3. Pruning occurs by restricting tree depth using max depth and by requiring each leaf to have at least min_samples_leaf. 

In [ ]:
def htree(xtn, ytn, xtst, ytst, ttl): 
    mx_dpth = list(range(1,31))
    f1tst = []; f1tn = []
    for m in mx_dpth:         
            clss = DecisionTreeClassifier(criterion="entropy", max_depth=m, random_state=7, min_samples_leaf=1)
            clss.fit(xtn, ytn)
            f1tst.append(f1_score(ytst, clss.predict(xtst)))
            f1tn.append(f1_score(ytn, clss.predict(xtn)))         
    ppt.plot(mx_dpth, f1tn, "o-", color = "y", label="Train")
    ppt.xlabel("Max Tree Depth")
    ppt.plot(mx_dpth, f1tst, "o-", color="m", label="Test")
    ppt.ylabel("F1 Score") 
    ppt.title(ttl)
    ppt.legend(loc="best")
    ppt.tight_layout()
    ppt.show()
     
    
def TreeGSCV(strt_lfn, end_lfn, xtn, ytn):
    param_grid = {"min_samples_leaf":my.linspace(strt_lfn,end_lfn,20).round().astype("int"), "max_depth":my.arange(1,20)}
    tr = GridSearchCV(param_grid=param_grid, estimator = DecisionTreeClassifier(), cv=10)
    tr.fit(xtn, ytn)
    print("Best Parameters -")
    print(tr.best_params_)
    return tr.best_params_["max_depth"], tr.best_params_["min_samples_leaf"]

In [ ]:
#BC
htree(bc_xtn, bc_ytn, bc_xtt, bc_ytt, ttl="Decision Tree Complexity Curve (Breast Cancer)\nHyperparameter : Tree Max Depth")

mx_dpth, mn_smpl_lf = TreeGSCV(round(0.005*len(bc_xtn)),round(0.05*len(bc_xtn)),bc_xtn,bc_ytn)

estmtr_bc = DecisionTreeClassifier(max_depth=mx_dpth, min_samples_leaf=mn_smpl_lf, random_state=7, criterion="entropy")

trn_sampsiz_bc, DT_tn_scr_bc, DT_ft_tm_bc, DT_prd_time_bc = pltlc(estmtr_bc, bc_xtn, bc_ytn, ttl="Decision Tree Breast Cancer")
class_eval(estmtr_bc, bc_xtn, bc_xtt, bc_ytn, bc_ytt)



# Titanic
htree(tt_xtn, tt_ytn, tt_xtt, tt_ytt, ttl="Decision Tree Complexity Curve (Titanic)\nHyperparameter : Tree Max Depth")

mx_dpth, mn_smpl_lf = TreeGSCV(round(0.005*len(tt_xtn)),round(0.05*len(tt_xtn)),tt_xtn,tt_ytn)

estmtr_tt = DecisionTreeClassifier(max_depth=mx_dpth, min_samples_leaf=mn_smpl_lf, random_state=7, criterion="entropy")

train_samp_tt, DT_tn_scr_tt, DT_ft_tm_tt, DT_prd_time_tt = pltlc(estmtr_tt, tt_xtn, tt_ytn, ttl="Decision Tree Titanic")
class_eval(estmtr_tt, tt_xtn, tt_xtt, tt_ytn, tt_ytt)


# Boosted Decision Tree

This section boosts the previous tree. Pruning will still be based on min_samples_leaf and max depth, but cutoff thresholds can be more smaller. The learning rate and estimators count are also introduced as hyperparameters which will determine each tree classifier's contribution.

In [ ]:
def hbst(xtn, ytn, xtst, ytst, mxdpth, mn_samples_lf, ttl):
    n_estmt = my.linspace(1,240,40).astype("int")
    f1tn = []; f1tst = []
    for n in n_estmt:         
            clss = GradientBoostingClassifier(min_samples_leaf=int(mn_samples_lf/2), n_estimators=n, max_depth=int(mxdpth/2), 
                                              random_state=7)
            clss.fit(xtn, ytn)
            f1tn.append(f1_score(ytn, clss.predict(xtn)))
            f1tst.append(f1_score(ytst, clss.predict(xtst)))
              
    ppt.plot(n_estmt, f1tst, "o-", color="m", label="Test")
    ppt.xlabel("No. Estimators")
    ppt.plot(n_estmt, f1tn, "o-", color = "y", label="Train")
    ppt.ylabel("F1 Score")
    ppt.title(ttl)
    ppt.legend(loc="best")
    ppt.tight_layout()
    ppt.show()

def BoostedGSCV(strt_lfn, end_lfn, xtn, ytn):
    param_grid = {"max_depth": my.arange(1,4),
                  "min_samples_leaf": my.linspace(strt_lfn,end_lfn,3).round().astype("int"),
                  "learning_rate": my.linspace(.001,.1,3),
                  "n_estimators": my.linspace(10,102,3).round().astype("int")}
    bst = GridSearchCV(param_grid=param_grid, estimator = GradientBoostingClassifier(), cv=10)
    bst.fit(xtn, ytn)
    print("Best Parameters -")
    print(bst.best_params_)
    return bst.best_params_["max_depth"], bst.best_params_["min_samples_leaf"], bst.best_params_["n_estimators"], bst.best_params_["learning_rate"]


In [ ]:
# BC
hbst(bc_xtn, bc_ytn, bc_xtt, bc_ytt, 3, 50, ttl="Boosted Tree Complexity Curve (Breast Cancer)\nHyperparameter : No. Estimators")

mx_dpth, mn_smpl_lf, bc_est, lrn_rt = BoostedGSCV(round(0.005*len(bc_xtn)),round(0.05*len(bc_xtn)),bc_xtn,bc_ytn)

estmtr_bc = GradientBoostingClassifier(learning_rate=lrn_rt, max_depth=mx_dpth, min_samples_leaf=mn_smpl_lf, 
                                              n_estimators=bc_est, random_state=7)

trn_sampsiz_bc, BT_tn_scr_bc, BT_ft_tm_bc, BT_prd_time_bc = pltlc(estmtr_bc, bc_xtn, bc_ytn, ttl="Boosted Tree Breast Cancer")

class_eval(estmtr_bc, bc_xtn, bc_xtt, bc_ytn, bc_ytt)


# Titanic
hbst(tt_xtn, tt_ytn, tt_xtt, tt_ytt, 3, 50, ttl="Boosted Tree Complexity Curve (Titanic)\nHyperparameter : No. Estimators")

mx_dpth, mn_smpl_lf, tt_est, lrn_rt = BoostedGSCV(round(0.005*len(tt_xtn)),round(0.05*len(tt_xtn)),tt_xtn,tt_ytn)

estmtr_tt = GradientBoostingClassifier(max_depth=mx_dpth, min_samples_leaf=mn_smpl_lf, 
                                              n_estimators=tt_est, learning_rate=lrn_rt, random_state=7)

trn_sampsiz_tt, BT_tn_scr_tt, BT_ft_tm_tt, BT_prd_time_tt = pltlc(estmtr_tt, tt_xtn, tt_ytn, ttl="Boosted Tree Titanic")

class_eval(estmtr_tt, tt_xtn, tt_xtt, tt_ytn, tt_ytt)


# KNN

This section builds a K-nearest neighbors classifier with a hyperparameter of n_neighbors.

In [ ]:
def hKNN(xtn, ytn, xtst, ytst, ttl):
    klst = my.linspace(1,200,25).astype("int")
    f1tn = []; f1tst = []
    for k in klst:
        clss = KNC(n_neighbors=k,n_jobs=-1)
        clss.fit(xtn,ytn)
        f1tst.append(f1_score(ytst, clss.predict(xtst)))
        f1tn.append(f1_score(ytn, clss.predict(xtn)))
    ppt.xlabel("No. Neighbors")    
    ppt.plot(klst, f1tst, "o-", color="m", label="Test")
    ppt.plot(klst, f1tn, "o-", color = "y", label="Train")
    ppt.ylabel("F1 Score")
    ppt.title(ttl)
    ppt.tight_layout()
    ppt.legend(loc="best")
    ppt.show()
    

In [ ]:
# BC
hKNN(bc_xtn, bc_ytn, bc_xtt, bc_ytt, ttl="kNN Complexity Curve (Breast Cancer)\nHyperparameter : No. Neighbors")

estmtr_bc = KNC(n_jobs=-1, n_neighbors=18)

trn_sampsiz_bc, kNN_tn_scr_bc, kNN_ft_tm_bc, kNN_prd_time_bc = pltlc(estmtr_bc, bc_xtn, bc_ytn, ttl="kNN Breast Cancer")
class_eval(estmtr_bc, bc_xtn, bc_xtt, bc_ytn, bc_ytt)



# Titanic
hKNN(tt_xtn, tt_ytn, tt_xtt, tt_ytt, ttl="kNN Complexity Curve (Titanic)\nHyperparameter : No. Neighbors")

estmtr_tt = KNC(n_jobs=-1, n_neighbors=10)

trn_sampsiz_tt, kNN_tn_scr_tt, kNN_ft_tm_tt, kNN_prd_time_tt = pltlc(estmtr_tt, tt_xtn, tt_ytn, ttl="kNN Titanic")
class_eval(estmtr_tt, tt_xtn, tt_xtt, tt_ytn, tt_ytt)


# Comparison Plots

This section plots learning rates and training times for the five different classifiers.

In [ ]:
def prd_tm(n,NNprd, SMVprd, kNNprd, DTprd, BTprd, ttl): 
    ppt.figure()
    ppt.title("Prediction: " + ttl)
    ppt.ylabel("Prediction [s]")
    ppt.plot(n, DTprd, "-", color="g", label="Decision Tree")
    ppt.plot(n, BTprd, "-", color="m", label="Boosted Tree")
    ppt.xlabel("Training Ex [#]")
    ppt.plot(n, NNprd, "-", color="k", label="Neural Net")
    ppt.plot(n, kNNprd, "-", color="r", label="kNN")
    ppt.plot(n, SMVprd, "-", color="b", label="SVM")
    ppt.legend(loc="best")
    ppt.show()

def prd_tm2(n,NNprd, SMVprd, DTprd, BTprd, ttl): 
    ppt.figure()
    ppt.title("Prediction: " + ttl)
    ppt.ylabel("Prediction [s]")
    ppt.plot(n, DTprd, "-", color="g", label="Decision Tree")
    ppt.plot(n, BTprd, "-", color="m", label="Boosted Tree")
    ppt.xlabel("Training Ex [#]")
    ppt.plot(n, NNprd, "-", color="k", label="Neural Net")
    ppt.plot(n, SMVprd, "-", color="b", label="SVM")
    ppt.legend(loc="best")
    ppt.show()

def ft_tm(n,NNtm, SMVtm, kNNtm, DTtm, BTtm, ttl):   
    ppt.figure()
    ppt.title("Training: " + ttl)
    ppt.ylabel("Training [s]")
    ppt.plot(n, DTtm, "-", color="g", label="Decision Tree")
    ppt.plot(n, BTtm, "-", color="m", label="Boosted Tree")
    ppt.xlabel("Training Ex [#]")
    ppt.plot(n, NNtm, "-", color="k", label="Neural Net")
    ppt.plot(n, kNNtm, "-", color="r", label="kNN")
    ppt.plot(n, SMVtm, "-", color="b", label="SVM")
    ppt.legend(loc="best")
    ppt.show()

def lrn_tm(n,NN, SVM, knn, DT, BT, ttl):
    ppt.figure()
    ppt.title("Learning Rates: " + ttl)
    ppt.ylabel("F1 Score")
    ppt.plot(n, DT, "-", color="g", label="Decision Tree")
    ppt.plot(n, BT, "-", color="m", label="Boosted Tree")
    ppt.xlabel("Training Ex [#]")
    ppt.plot(n, NN, "-", color="k", label="Neural Net")
    ppt.plot(n, knn, "-", color="r", label="kNN")
    ppt.plot(n, SVM, "-", color="b", label="SVM")
    ppt.legend(loc="best")
    ppt.show()

In [ ]:
ft_tm(trn_sampsiz_bc, NN_ft_tm_bc, SVM_ft_tm_bc, kNN_ft_tm_bc, 
                 DT_ft_tm_bc, BT_ft_tm_bc, "Breast Cancer")              
prd_tm(trn_sampsiz_bc, NN_prd_time_bc, SVM_prd_time_bc, kNN_prd_time_bc, 
                 DT_prd_time_bc, BT_prd_time_bc, "Breast Cancer")   
prd_tm2(trn_sampsiz_bc, NN_prd_time_bc, SVM_prd_time_bc, 
                 DT_prd_time_bc, BT_prd_time_bc, "Breast Cancer")   
lrn_tm(trn_sampsiz_bc, NN_tn_scr_bc, SVM_tn_scr_bc, kNN_tn_scr_bc, 
                 DT_tn_scr_bc, BT_tn_scr_bc, "Breast Cancer")  



ft_tm(trn_sampsiz_tt, NN_ft_tm_tt, SVM_ft_tm_tt, kNN_ft_tm_tt, 
                 DT_ft_tm_tt, BT_ft_tm_tt, "Titanic")       
prd_tm(trn_sampsiz_tt, NN_prd_time_tt, SVM_prd_time_tt, kNN_prd_time_tt, 
                 DT_prd_time_tt, BT_prd_time_tt, "Titanic") 
prd_tm2(trn_sampsiz_tt, NN_prd_time_tt, SVM_prd_time_tt, 
                 DT_prd_time_tt, BT_prd_time_tt, "Titanic") 
lrn_tm(trn_sampsiz_tt, NN_tn_scr_tt, SVM_tn_scr_tt, kNN_tn_scr_tt, 
                 DT_tn_scr_tt, BT_tn_scr_tt, "Titanic")